In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
nfhs4 = pd.read_csv("../data/raw/NFHS-4.csv")
nfhs5 = pd.read_csv("../data/raw/NFHS-5.csv")

print("NFHS4:", nfhs4.shape)
print("NFHS5:", nfhs5.shape)

NFHS4: (1911, 98)
NFHS5: (704, 108)


In [5]:
nfhs4 = nfhs4[
    nfhs4["Residence Type"] == "Total"
].copy()

In [6]:
nfhs4.shape

(637, 98)

In [7]:
NFHS4_MAP = {
    "STUNTING":
    "Children Under 5 Years Who Are Stunted (Height-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1",

    "WASTING":
    "Children Under 5 Years Who Are Wasted (Weight-For-Height) (%) (UOM:%(Percentage)), Scaling Factor:1",

    "UNDERWEIGHT":
    "Children Under 5 Years Who Are Underweight (Weight-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1",

    "ANEMIA":
    "Children Age Group 6 To 59 Months Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1",

    "INST_DEL":
    "Institutional Births (%) (UOM:%(Percentage)), Scaling Factor:1",

    "SANITATION":
    "Population Living In Households That Use An Improved Sanitation Facility (%) (UOM:%(Percentage)), Scaling Factor:1",

    "CLEAN_FUEL":
    "Households Using Clean Fuel For Cooking (%) (UOM:%(Percentage)), Scaling Factor:1",

    "IMMUNIZATION":
    "Children In The Age Group Of 12 To 23 Months Who Are Fully Immunized (Bacille Calmette-Guerin (Bcg), Measles, And 3 Doses Each Of Polio And Dpt) (%) (UOM:%(Percentage)), Scaling Factor:1"
}

In [8]:
NFHS5_MAP = {
    "STUNTING":
    "Children Under 5 Years Who Are Stunted (Height-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1",

    "WASTING":
    "Children Under 5 Years Who Are Wasted (Weight-For-Height) (%) (UOM:%(Percentage)), Scaling Factor:1",

    "UNDERWEIGHT":
    "Children Under 5 Years Who Are Underweight (Weight-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1",

    "ANEMIA":
    "Children Age Group 6 To 59 Months Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1",

    "INST_DEL":
    "Institutional Births (%) (UOM:%(Percentage)), Scaling Factor:1",

    "SANITATION":
    "Population Living In Households That Use An Improved Sanitation Facility (%) (UOM:%(Percentage)), Scaling Factor:1",

    "CLEAN_FUEL":
    "Households Using Clean Fuel For Cooking (%) (UOM:%(Percentage)), Scaling Factor:1",

    "IMMUNIZATION":
    "Children Age Group 12 To 23 Months Fully Vaccinated Based On Information From Either Vaccination Card Or Mothers Recall (%) (UOM:%(Percentage)), Scaling Factor:1"
}

In [9]:
for k, v in NFHS4_MAP.items():
    print(k, "->", v in nfhs4.columns)

print("-" * 50)

for k, v in NFHS5_MAP.items():
    print(k, "->", v in nfhs5.columns)

STUNTING -> True
WASTING -> True
UNDERWEIGHT -> True
ANEMIA -> True
INST_DEL -> True
SANITATION -> True
CLEAN_FUEL -> True
IMMUNIZATION -> True
--------------------------------------------------
STUNTING -> True
WASTING -> True
UNDERWEIGHT -> True
ANEMIA -> True
INST_DEL -> True
SANITATION -> True
CLEAN_FUEL -> True
IMMUNIZATION -> True


In [10]:
nfhs4_clean = nfhs4[
    ["State", "District"] + list(NFHS4_MAP.values())
].copy()

nfhs4_clean.columns = [
    "State",
    "District",
    *NFHS4_MAP.keys()
]

nfhs4_clean.shape

(637, 10)

In [11]:
nfhs5_clean = nfhs5[
    ["State", "District"] + list(NFHS5_MAP.values())
].copy()

nfhs5_clean.columns = [
    "State",
    "District",
    *NFHS5_MAP.keys()
]

nfhs5_clean.shape

(704, 10)

In [12]:
nfhs4_clean[
    nfhs4_clean["District"] == "Unknown Districts Of India"
]

,State,District,STUNTING,WASTING,UNDERWEIGHT,ANEMIA,INST_DEL,SANITATION,CLEAN_FUEL,IMMUNIZATION
1070,Maharashtra,Unknown Districts Of India,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
nfhs4_clean = nfhs4_clean[
    nfhs4_clean["District"] != "Unknown Districts Of India"
].copy()

In [14]:
nfhs4_clean.to_csv(
    "../data/interim/nfhs4_clean.csv",
    index=False
)

nfhs5_clean.to_csv(
    "../data/interim/nfhs5_clean.csv",
    index=False
)

print("Saved")

Saved


## NFHS Cleaning Summary

Raw NFHS4:
1911 × 98

Raw NFHS5:
704 × 108

Actions:
1. Filtered NFHS4 to Residence Type = Total
2. Removed "Unknown Districts Of India"
3. Selected 8 welfare outcomes
4. Standardized column names

Outputs:
- nfhs4_clean.csv (636 districts)
- nfhs5_clean.csv (704 districts)

Known Missing Values:
- NFHS4 IMMUNIZATION: 5 districts
- NFHS5 IMMUNIZATION: 5 districts

Next Block: SHRUG Feature Engineering

In [15]:
loc = pd.read_csv("../data/raw/shrid_loc_names.csv")
key = pd.read_csv("../data/raw/shrid_pc11dist_key.csv")
pca = pd.read_csv("../data/raw/pc11_pca_clean_pc11dist.csv")

print("LOC :", loc.shape)
print("KEY :", key.shape)
print("PCA :", pca.shape)

LOC : (596389, 7)
KEY : (596508, 3)
PCA : (640, 87)


In [16]:
shrug_features = pca[
    [
        "pc11_state_id",
        "pc11_district_id",
        "pc11_pca_tot_p",
        "pc11_pca_no_hh",
        "pc11_pca_p_sc",
        "pc11_pca_p_st",
        "pc11_pca_f_lit",
        "pc11_pca_tot_f",
        "pc11_pca_main_al_p"
    ]
].copy()

In [17]:
shrug_features["female_literacy_pct"] = (
    shrug_features["pc11_pca_f_lit"]
    / shrug_features["pc11_pca_tot_f"]
) * 100

shrug_features["scst_pct"] = (
    (
        shrug_features["pc11_pca_p_sc"]
        + shrug_features["pc11_pca_p_st"]
    )
    / shrug_features["pc11_pca_tot_p"]
) * 100

shrug_features["agri_worker_pct"] = (
    shrug_features["pc11_pca_main_al_p"]
    / shrug_features["pc11_pca_tot_p"]
) * 100

In [18]:
shrug_features = shrug_features.rename(
    columns={
        "pc11_pca_tot_p": "total_population",
        "pc11_pca_no_hh": "total_households"
    }
)

In [19]:
shrug_features = shrug_features[
    [
        "pc11_state_id",
        "pc11_district_id",
        "female_literacy_pct",
        "scst_pct",
        "agri_worker_pct",
        "total_population",
        "total_households"
    ]
]

In [20]:
print(shrug_features.shape)

print(
    shrug_features.isna().sum()
)

shrug_features.describe()

(640, 7)
pc11_state_id          0
pc11_district_id       0
female_literacy_pct    0
scst_pct               0
agri_worker_pct        0
total_population       0
total_households       0
dtype: int64


,pc11_state_id,pc11_district_id,female_literacy_pct,scst_pct,agri_worker_pct,total_population,total_households
count,640.000000,640.000000,640.000000,640.000000,640.000000,6.400000e+02,6.400000e+02
mean,17.114062,320.500000,55.231263,32.573469,6.312803,1.891784e+06,3.898089e+05
std,9.426486,184.896367,12.410804,22.578881,4.965442,1.544009e+06,3.374620e+05
min,1.000000,1.000000,24.245409,0.344366,0.000000,8.004000e+03,1.952000e+03
25%,9.000000,160.750000,45.852929,17.986011,2.321014,8.178610e+05,1.694018e+05
50%,18.000000,320.500000,54.654506,25.118480,4.999728,1.557367e+06,3.141715e+05
75%,24.000000,480.250000,64.303532,37.794701,9.123231,2.583551e+06,5.340748e+05
max,35.000000,640.000000,88.622864,98.575090,23.856051,1.106015e+07,2.529165e+06


In [21]:
agri_worker_pct = (
    pca["pc11_pca_main_al_p"]
    / pca["pc11_pca_mainwork_p"]
) * 100

In [22]:
shrug_features = pd.DataFrame()

shrug_features["pc11_state_id"] = pca["pc11_state_id"]
shrug_features["pc11_district_id"] = pca["pc11_district_id"]

shrug_features["female_literacy_pct"] = (
    pca["pc11_pca_f_lit"]
    / pca["pc11_pca_tot_f"]
) * 100

shrug_features["scst_pct"] = (
    (
        pca["pc11_pca_p_sc"]
        + pca["pc11_pca_p_st"]
    )
    / pca["pc11_pca_tot_p"]
) * 100

shrug_features["agri_worker_pct"] = (
    pca["pc11_pca_main_al_p"]
    / pca["pc11_pca_mainwork_p"]
) * 100

shrug_features["total_population"] = pca["pc11_pca_tot_p"]

shrug_features["total_households"] = pca["pc11_pca_no_hh"]

In [23]:
shrug_features.to_csv(
    "../data/interim/shrug_features.csv",
    index=False
)

print("shrug_features saved")

shrug_features saved


In [25]:
poverty = pd.read_csv(
    "../data/raw/secc_cons_rural_pc11dist.csv"
)

urban = pd.read_csv(
    "../data/raw/secc_cons_urban_pc11dist.csv"
)

print("Rural:", poverty.shape)
print("Urban:", urban.shape)

print(poverty.columns.tolist())
print("-" * 100)
print(urban.columns.tolist())

Rural: (615, 10)
Urban: (594, 10)
['pc11_district_id', 'pc11_state_id', 'secc_cons_rural', 'secc_cons_pc_rural', 'secc_pov_rate_rural', 'secc_pov_rate_tend_rural', '_mean_p_miss', '_core_p_miss', '_target_weight_share', '_target_group_max_weight_share']
----------------------------------------------------------------------------------------------------
['pc11_district_id', 'pc11_state_id', 'secc_cons_urban', 'secc_cons_pc_urban', 'secc_pov_rate_urban', 'secc_pov_rate_tend_urban', '_mean_p_miss', '_core_p_miss', '_target_weight_share', '_target_group_max_weight_share']


In [26]:
print(poverty["pc11_district_id"].nunique())
print(urban["pc11_district_id"].nunique())

615
594


In [27]:
print(
    set(poverty["pc11_district_id"])
    - set(urban["pc11_district_id"])
)

print(
    set(urban["pc11_district_id"])
    - set(poverty["pc11_district_id"])
)

{391, 394, 638, 141, 272, 273, 274, 279, 25, 542, 289, 34, 290, 291, 292, 317, 318, 324, 332, 335, 341, 343, 489, 374, 380, 381, 382, 639}
{518, 519, 496, 536, 603, 94, 95}


In [28]:
poverty_merged = poverty[
    ["pc11_district_id", "pc11_state_id", "secc_pov_rate_rural"]
].merge(
    urban[
        ["pc11_district_id", "secc_pov_rate_urban"]
    ],
    on="pc11_district_id",
    how="outer"
)

In [29]:
poverty_merged["poverty_rate"] = (
    poverty_merged[
        [
            "secc_pov_rate_rural",
            "secc_pov_rate_urban"
        ]
    ]
    .mean(axis=1)
)

In [30]:
poverty_merged["poverty_log"] = np.log1p(
    poverty_merged["poverty_rate"]
)

In [31]:
poverty_features = poverty_merged[
    [
        "pc11_state_id",
        "pc11_district_id",
        "poverty_rate",
        "poverty_log"
    ]
]

In [32]:
print(poverty_features.shape)

poverty_features.isna().sum()


(622, 4)


pc11_state_id       7
pc11_district_id    0
poverty_rate        0
poverty_log         0
dtype: int64

In [33]:
poverty_features.to_csv(
    "../data/interim/poverty_features.csv",
    index=False
)

print("poverty_features saved")

poverty_features saved


Poverty Features Notes

- 622 district records generated.
- Poverty rate computed as mean of available rural and urban poverty rates.
- poverty_log = log1p(poverty_rate).
- 7 districts have missing pc11_state_id because they exist only in the urban SECC file.
- Poverty values are present for all districts.
- State IDs will be recovered during master dataset merging.

In [34]:
poverty_features["poverty_log"].describe()

count    622.000000
mean       0.199214
std        0.091312
min        0.010549
25%        0.120865
50%        0.209129
75%        0.268820
max        0.524568
Name: poverty_log, dtype: float64

In [35]:
poverty_features.to_csv(
    "../data/interim/poverty_features.csv",
    index=False
)

print("poverty_features saved")

poverty_features saved


In [36]:
night = pd.read_csv(
    "../data/raw/viirs_annual_pc11dist.csv"
)

print(night.shape)

for c in night.columns:
    print(c)

(15360, 9)
pc11_district_id
pc11_state_id
viirs_annual_min
viirs_annual_max
viirs_annual_mean
viirs_annual_sum
viirs_annual_num_cells
category
year


In [37]:
night[
    [
        "viirs_annual_mean",
        "viirs_annual_sum",
        "year"
    ]
].head()

,viirs_annual_mean,viirs_annual_sum,year
0,0.506407,40484.480053,2021
1,0.901833,35284.502309,2021
2,1.092501,63880.362758,2021
3,1.526908,72835.408423,2021
4,43.376884,49900.414293,2021


In [38]:
night_2021 = night[
    (night["year"] == 2021)
    &
    (night["category"] == "average-masked")
].copy()

night_2021.shape

(640, 9)

In [39]:
nightlight_features = night_2021[
    [
        "pc11_state_id",
        "pc11_district_id",
        "viirs_annual_mean"
    ]
].copy()

nightlight_features["night_lights_log"] = np.log1p(
    nightlight_features["viirs_annual_mean"]
)

nightlight_features = nightlight_features[
    [
        "pc11_state_id",
        "pc11_district_id",
        "night_lights_log"
    ]
]

In [40]:
nightlight_features.to_csv(
    "../data/interim/nightlight_features.csv",
    index=False
)

print("nightlight_features saved")

nightlight_features saved


In [41]:
m1 = pd.read_csv("../data/raw/2019-20.csv")
m2 = pd.read_csv("../data/raw/2020-21.csv")
m3 = pd.read_csv("../data/raw/2021-22.csv")

print(m1.shape)
print(m2.shape)
print(m3.shape)

(8052, 36)
(8544, 36)
(8592, 36)


In [42]:
m1[
    [
        "district_code",
        "district_name",
        "Average_days_of_employment_provided_per_Household",
        "Women_Persondays",
        "percentage_payments_gererated_within_15_days"
    ]
].head()

,district_code,district_name,Average_days_of_employment_provided_per_Household,Women_Persondays,percentage_payments_gererated_within_15_days
0,103,NORTH AND MIDDLE ANDAMAN,42,102294,596.12
1,103,NORTH AND MIDDLE ANDAMAN,43,105714,564.61
2,102,NICOBARS,21,263,0.00
3,102,NICOBARS,21,263,0.00
4,101,SOUTH ANDAMAN,16,11052,278.04


In [43]:
mgnregs = pd.concat(
    [m1, m2, m3],
    ignore_index=True
)

print(mgnregs.shape)
print(mgnregs["district_code"].nunique())

(25188, 36)
716


In [44]:
mgnregs[
    "percentage_payments_gererated_within_15_days"
] = (
    mgnregs[
        "percentage_payments_gererated_within_15_days"
    ]
    .clip(upper=100)
)

In [45]:
mgnregs["wage_timeliness_pct"] = (
    mgnregs["percentage_payments_gererated_within_15_days"]
)

mgnregs["avg_days_per_hh"] = (
    mgnregs["Average_days_of_employment_provided_per_Household"]
)

mgnregs["women_pct"] = (
    mgnregs["Women_Persondays"]
    /
    mgnregs["Persondays_of_Central_Liability_so_far"]
) * 100

mgnregs["persondays_per_hh"] = (
    mgnregs["Persondays_of_Central_Liability_so_far"]
    /
    mgnregs["Total_Households_Worked"]
)

In [46]:
mgnregs["women_pct"] = mgnregs["women_pct"].fillna(0)

mgnregs["persondays_per_hh"] = (
    mgnregs["persondays_per_hh"]
    .fillna(0)
)

In [53]:
mgnregs_features = (
    mgnregs
    .groupby(
        ["state_name","district_code", "district_name"],
        as_index=False
    )
    [
        [
            "wage_timeliness_pct",
            "avg_days_per_hh",
            "women_pct",
            "persondays_per_hh"
        ]
    ]
    .mean()
)

In [48]:
mgnregs_features.to_csv(
    "../data/interim/mgnregs_features.csv",
    index=False
)

print(mgnregs_features.shape)

(716, 7)


In [49]:
print("NFHS4:", nfhs4_clean.shape)
print("NFHS5:", nfhs5_clean.shape)

print("SHRUG:", shrug_features.shape)

print("POVERTY:", poverty_features.shape)

print("NIGHTLIGHT:", nightlight_features.shape)

print("MGNREGS:", mgnregs_features.shape)

NFHS4: (636, 10)
NFHS5: (704, 10)
SHRUG: (640, 7)
POVERTY: (622, 4)
NIGHTLIGHT: (640, 3)
MGNREGS: (716, 7)


In [51]:
mgnregs_features.to_csv(
    "../data/interim/mgnregs_features.csv",
    index=False
)

In [54]:
mgnregs.groupby("state_name")["district_code"].agg(
    ["min", "max", "nunique"]
).head(20)

,min,max,nunique
state_name,,,
ANDAMAN AND NICOBAR,101,103,3
ANDHRA PRADESH,201,213,13
ARUNACHAL PRADESH,301,326,25
ASSAM,401,434,33
BIHAR,501,551,38
CHHATTISGARH,3301,3328,28
DN HAVELI AND DD,701,701,1
GOA,1001,1002,2
GUJARAT,1101,1133,33


In [55]:
mgnregs[
    ["state_name", "district_code", "district_name"]
].sort_values(
    ["state_name", "district_code"]
).head(100)

,state_name,district_code,district_name
4,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
7,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
8,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
13,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
16,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
17,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
19,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
21,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
22,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
26,ANDAMAN AND NICOBAR,101,SOUTH ANDAMAN
